# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 7: KPI Engine & Business Recommendations

**Objective:** Calculate all business KPIs, create a summary dashboard, and provide actionable recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs('../Images/Phase7', exist_ok=True)

LAUNCH_DATE = pd.Timestamp('2024-06-01')
LAUNCHED_PRODUCTS = ['P1', 'P2', 'P3', 'P4', 'P5']

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../Data/cannibalization_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

before = df[df['Period_Flag'] == 'Before_Launch']
after  = df[df['Period_Flag'] == 'After_Launch']

## 1. KPI Calculations

In [ ]:
# 1. Total Sales
total_sales = df['Sales'].sum()

# 2. Total Revenue
total_revenue = df['Revenue'].sum()

# 3. Average Selling Price
avg_price = df['Price'].mean()

# 4. Sales Growth %
avg_sales_before = before['Sales'].mean()
avg_sales_after  = after['Sales'].mean()
sales_growth = ((avg_sales_after - avg_sales_before) / avg_sales_before * 100)

# 5. Revenue Growth %
avg_rev_before = before['Revenue'].mean()
avg_rev_after  = after['Revenue'].mean()
rev_growth = ((avg_rev_after - avg_rev_before) / avg_rev_before * 100)

# 6. Cannibalization Rate
months_before = before['Year_Month'].nunique()
months_after  = after['Year_Month'].nunique()
p6_b = before[before['Product_ID'] == 'P6']['Sales'].sum()
p6_a = after[after['Product_ID'] == 'P6']['Sales'].sum()
p6_loss = (p6_b / months_before) * months_after - p6_a
p4p5_gain = after[after['Product_ID'].isin(['P4', 'P5'])]['Sales'].sum()
cann_rate = (p6_loss / p4p5_gain * 100) if p4p5_gain > 0 else 0

# 7. Market Share (P6 share before vs after)
p6_share_before = (before[before['Product_ID'] == 'P6']['Revenue'].sum() / before['Revenue'].sum() * 100)
p6_share_after  = (after[after['Product_ID'] == 'P6']['Revenue'].sum() / after['Revenue'].sum() * 100)

# 8. Average Rating
avg_rating = df['Rating'].mean()

# 9. Customer Retention (customers who bought before AND after)
cust_before = set(before['Customer_ID'])
cust_after  = set(after['Customer_ID'])
cust_retained = cust_before & cust_after
retention_rate = len(cust_retained) / len(cust_before) * 100 if cust_before else 0

# 10. Marketing ROI
mkt_roi = df['Revenue'].sum() / df['Marketing_Spend'].sum()

# 11. Stock Availability %
stock_avail = (df['Stock_Available'].mean() * 100)

# 12. Product Contribution (top product's revenue share)
top_prod_contrib = (df.groupby('Product_ID')['Revenue'].sum().max() / total_revenue * 100)

print("KPIs calculated successfully.")

## 2. KPI Summary Dashboard

In [ ]:
kpi_table = pd.DataFrame({
    'KPI': [
        'Total Sales',
        'Total Revenue',
        'Avg Selling Price',
        'Sales Growth %',
        'Revenue Growth %',
        'Cannibalization Rate',
        'P6 Market Share (Before)',
        'P6 Market Share (After)',
        'Average Rating',
        'Customer Retention Rate',
        'Marketing ROI',
        'Stock Availability %',
        'Top Product Contribution %'
    ],
    'Value': [
        f'{total_sales:,} units',
        f'{total_revenue:,.0f}',
        f'{avg_price:.2f}',
        f'{sales_growth:.2f}%',
        f'{rev_growth:.2f}%',
        f'{cann_rate:.1f}%',
        f'{p6_share_before:.2f}%',
        f'{p6_share_after:.2f}%',
        f'{avg_rating:.2f}',
        f'{retention_rate:.1f}%',
        f'{mkt_roi:.2f}',
        f'{stock_avail:.1f}%',
        f'{top_prod_contrib:.2f}%'
    ]
})

print("=" * 55)
print("  KPI SUMMARY DASHBOARD")
print("=" * 55)
for _, row in kpi_table.iterrows():
    print(f"  {row['KPI']:<30s} : {row['Value']}")
print("=" * 55)

## 3. KPI Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Sales Before vs After
ax = axes[0, 0]
ax.bar(['Before', 'After'], [before['Sales'].sum(), after['Sales'].sum()], color=['#74b9ff', '#55efc4'])
ax.set_title('Total Sales', fontweight='bold')

# 2. Revenue Before vs After
ax = axes[0, 1]
ax.bar(['Before', 'After'], [before['Revenue'].sum(), after['Revenue'].sum()], color=['#74b9ff', '#55efc4'])
ax.set_title('Total Revenue', fontweight='bold')

# 3. Cannibalization Rate Gauge (simple bar)
ax = axes[0, 2]
ax.barh(['Cann Rate'], [cann_rate], color='red', height=0.4)
ax.set_xlim(0, 100)
ax.set_title(f'Cannibalization Rate: {cann_rate:.1f}%', fontweight='bold')

# 4. Customer Retention
ax = axes[1, 0]
ax.pie([retention_rate, 100-retention_rate], labels=['Retained', 'New/Lost'],
       colors=['#55efc4', '#dfe6e9'], autopct='%1.1f%%', startangle=90)
ax.set_title('Customer Retention', fontweight='bold')

# 5. Stock Availability
ax = axes[1, 1]
ax.pie([stock_avail, 100-stock_avail], labels=['In Stock', 'OOS'],
       colors=['#55efc4', '#ff7675'], autopct='%1.1f%%', startangle=90)
ax.set_title('Stock Availability', fontweight='bold')

# 6. Marketing ROI
ax = axes[1, 2]
mkt_by_period = pd.Series({
    'Before': before['Revenue'].sum() / before['Marketing_Spend'].sum(),
    'After': after['Revenue'].sum() / after['Marketing_Spend'].sum()
})
ax.bar(mkt_by_period.index, mkt_by_period.values, color=['#74b9ff', '#55efc4'])
ax.set_title('Marketing ROI', fontweight='bold')

plt.suptitle('KPI Dashboard Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Images/Phase7/01_kpi_dashboard.png', bbox_inches='tight')
plt.show()

## 4. Before vs After — Complete Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Avg Sales', 'Avg Revenue', 'Avg Price', 'Avg Rating', 'Marketing ROI', 'Stock %'],
    'Before Launch': [
        f"{before['Sales'].mean():.2f}",
        f"{before['Revenue'].mean():.2f}",
        f"{before['Price'].mean():.2f}",
        f"{before['Rating'].mean():.2f}",
        f"{before['Revenue'].sum()/before['Marketing_Spend'].sum():.2f}",
        f"{before['Stock_Available'].mean()*100:.1f}%"
    ],
    'After Launch': [
        f"{after['Sales'].mean():.2f}",
        f"{after['Revenue'].mean():.2f}",
        f"{after['Price'].mean():.2f}",
        f"{after['Rating'].mean():.2f}",
        f"{after['Revenue'].sum()/after['Marketing_Spend'].sum():.2f}",
        f"{after['Stock_Available'].mean()*100:.1f}%"
    ]
})

print("Before vs After Launch — Comparison:")
print(comparison.to_string(index=False))

## 5. Business Recommendations

In [ ]:
recommendations = [
    ("Product Strategy",
     "Monitor P6 closely — it shows significant cannibalization from P4/P5. "
     "Consider differentiating P6 or phasing it out if decline continues."),

    ("Pricing Strategy",
     "P6 may benefit from price repositioning to compete with P4/P5. "
     "Consider promotional pricing to retain existing P6 customers."),

    ("Marketing Focus",
     "Redirect marketing spend to support P6 retention while promoting P4/P5 to new customer segments "
     "rather than poaching existing P6 customers."),

    ("Inventory Planning",
     "Maintain strong stock availability for all products. "
     "Stockouts compound the cannibalization effect by pushing customers to alternatives."),

    ("Customer Retention",
     "Implement loyalty programs for P6 customers to reduce switching. "
     "Target switchers with win-back campaigns."),

    ("Launch Strategy",
     "For future launches, conduct cannibalization impact assessment BEFORE launch. "
     "Ensure new products target NEW customer segments to avoid internal competition."),

    ("Regional Strategy",
     "Performance is balanced across regions. No region-specific interventions needed. "
     "Continue uniform distribution strategy."),

    ("Growth Assessment",
     "Overall growth may be misleading if driven by new products cannibalizing existing ones. "
     "Measure NET new revenue (total growth minus cannibalized revenue) for accurate assessment.")
]

print("=" * 60)
print("  BUSINESS RECOMMENDATIONS")
print("=" * 60)
for i, (area, rec) in enumerate(recommendations, 1):
    print(f"\n  {i}. {area}")
    print(f"     {rec}")
print("\n" + "=" * 60)

## 6. Save Final Analysis-Ready Dataset

In [ ]:
# Save the complete dataset
output_path = "../Data/cannibalization_final.csv"
df.to_csv(output_path, index=False)

print(f"Final dataset saved to: {output_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nAll {df.shape[1]} columns:")
for col in df.columns:
    print(f"  - {col}")

print("\n" + "=" * 60)
print("  PROJECT COMPLETE")
print("=" * 60)
print("  All 7 phases have been executed successfully.")
print("  Deliverables:")
print("    - Clean Dataset      : Data/cannibalization_cleaned.csv")
print("    - Final Dataset      : Data/cannibalization_final.csv")
print("    - EDA Charts         : Images/Phase3-7/")
print("    - KPI Dashboard      : Phase 7 notebook")
print("    - Recommendations    : Phase 7 notebook")
print("=" * 60)